# 第6章：几何变换

## 编程实践：手写相似 / 仿射 / 单应变换与 warp

| 项目 | 说明 |
|------|------|
| 输入图片 | `lena.jpeg`（来自 Hands-on-CV（上海交通大学《动手学习计算机视觉》）参考资料第 2/5 章） |
| 手写核心 | 变换矩阵构造、逆映射、双线性插值 warp、白底拼接 |
| 允许调用 | 仅 `cv_imread` / `cv_imwrite` 图像读写 |
| 对比验证 | 与 OpenCV `cv2.warpAffine` / `cv2.warpPerspective` 做数值误差对比 |


## 一、学习目标

### 🎯 什么是几何变换？——零基础的直觉

**几何变换 = 不改变像素的颜色值，只改变它们"在画布上的位置"。**

就像你把一张照片：
- 旋转 45°；
- 放大缩小；
- 向右平移 50 像素；
- 用手"拉扯"四个角让它变成梯形（透视变形）；

这些全都是几何变换。**变换的核心是数学上的矩阵乘法 + 齐次坐标。**

---

### 📖 具体学习目标
1. 掌握**相似变换**（旋转 + 平移 + 缩放）、**仿射变换**、**单应变换**的实现方法。
2. 理解**前向映射**与**反向映射（逆映射）**的区别，以及为什么实现 warp 通常用反向映射。
3. 掌握**双线性插值**的原理与手写实现。

---

### 三种几何变换一张表搞懂（齐次坐标下的矩阵）

| 变换类型 | 矩阵形式 | 自由度 | 保持什么不变？ | 生活类比 |
|----------|----------|:------:|----------------|----------|
| **相似变换** | $\begin{bmatrix} s\cos\theta & -s\sin\theta & t_x \\ s\sin\theta & s\cos\theta & t_y \end{bmatrix}$ | 4（s, θ, tx, ty） | 角度、形状、平行线 | 把照片缩放+旋转+贴到相册里 |
| **仿射变换** | $\begin{bmatrix} a & b & t_x \\ c & d & t_y \end{bmatrix}$ | 6 | 平行线、平行性、共线性 | 把照片贴到玻璃门上然后斜着看 |
| **单应变换（透视）** | 3×3 满秩矩阵 $\mathbf{H}$（最后一行一般不为 0） | 8 | 共线性、交比 | 把照片贴到地面上看（近大远小） |

**齐次坐标小提示（零基础必看）：**
> 为什么要在 $(x, y)$ 后面加个 1 变成 $(x, y, 1)$？
> 因为**平移不是线性变换**，不能用 2×2 矩阵乘出来——但升级到齐次坐标后，
> 旋转、缩放、平移、透视全都可以统一成**一次矩阵乘法**，代码写起来特别干净。

---

### 为什么 warp 用反向映射 + 双线性插值？（零基础必懂）

**❌ 前向映射（"把原图上的点挪到新位置"）有两个致命问题：**
1. **空洞**：多个原图像素映射到同一个输出像素（重叠），或者有的输出像素没人认领（黑洞）。
2. **浮点位置**：原图的整数坐标映射到输出图通常是"半个像素的位置"，不知道该放哪。

**✅ 反向映射（"从输出图倒着问每个像素该找谁要颜色"）完美解决：**
1. 遍历输出图的每个整数坐标 $(x', y')$；
2. 用**逆矩阵**反推它对应原图里哪个浮点坐标 $(x, y)$；
3. 对 $(x, y)$ 周围 4 个整数像素做**双线性插值**得到一个平滑值。

这样保证输出图每个格子都有颜色、且不会重叠，结果干净利落。

---

### 双线性插值："在四个邻居之间公平地加权平均"

设浮点坐标 $(x + u, y + v)$，其中 $u, v \in [0,1)$ 是小数部分，四个邻居是：

$$
\begin{matrix}
\text{top-left } I[y, x] & \text{top-right } I[y, x+1] \\
\text{bottom-left } I[y+1, x] & \text{bottom-right } I[y+1, x+1]
\end{matrix}
$$

分两步：
1. 先在水平方向插值两次 → top / bottom 两个中间值；
2. 再在垂直方向插值一次 → 最终结果。

$$
I_{out} = (1-u)(1-v)I_{tl} + u(1-v)I_{tr} + (1-u)v I_{bl} + u v I_{br}
$$

> 🎓 **一句话记忆**：权重 = 距离的反比——离哪个邻居越近，那个邻居话语权越大。



## 二、手写约束清单

- ✅ 允许：`cv_imread` / `cv_imwrite`；Python 循环与算术；`np.zeros` 开辟空间。
- ❌ 禁止：`cv2.warpAffine` / `cv2.warpPerspective` / `cv2.getRotationMatrix2D` 用于实现（仅可用于对比验证）。
- 变换矩阵参数由代码显式构造（自行设定）。


In [ ]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


In [ ]:
import math


def make_similarity_matrix(scale, angle_deg, tx, ty):
    """构造相似变换矩阵（2x3）。"""
    a = math.radians(angle_deg)
    return np.array([
        [scale * math.cos(a), -scale * math.sin(a), tx],
        [scale * math.sin(a),  scale * math.cos(a), ty],
    ], dtype=np.float64)


def make_affine_matrix(a, b, c, d, tx, ty):
    """构造仿射变换矩阵（2x3）。"""
    return np.array([[a, b, tx], [c, d, ty]], dtype=np.float64)


def make_homography_matrix(h11, h12, h13, h21, h22, h23, h31, h32, h33):
    """构造单应变换矩阵（3x3）。"""
    return np.array([[h11, h12, h13], [h21, h22, h23], [h31, h32, h33]], dtype=np.float64)


def invert_matrix(M):
    """手写 2x2 / 3x3 矩阵求逆（用伴随矩阵，避免调用 np.linalg.inv）。"""
    M = np.array(M, dtype=np.float64)
    if M.shape == (2, 2):
        det = M[0, 0] * M[1, 1] - M[0, 1] * M[1, 0]
        return (1.0 / det) * np.array([[M[1, 1], -M[0, 1]], [-M[1, 0], M[0, 0]]])
    # 3x3 伴随矩阵法
    def cofactor(i, j):
        sub = np.delete(np.delete(M, i, axis=0), j, axis=1)
        return ((-1) ** (i + j)) * (sub[0, 0] * sub[1, 1] - sub[0, 1] * sub[1, 0])
    cof = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            cof[i, j] = cofactor(i, j)
    det = M[0, 0] * cof[0, 0] + M[0, 1] * cof[0, 1] + M[0, 2] * cof[0, 2]
    return cof.T / det


def warp_homography_manual(image, H, out_w, out_h, border_value=255):
    """手写单应 warp：反向映射 + 双线性插值，输出白底图。"""
    H_inv = invert_matrix(H)
    out = np.full((out_h, out_w, image.shape[2]), border_value, dtype=np.uint8)

    for y in range(out_h):
        for x in range(out_w):
            # 反向映射：输出坐标 -> 输入齐次坐标
            sx = H_inv[0, 0] * x + H_inv[0, 1] * y + H_inv[0, 2]
            sy = H_inv[1, 0] * x + H_inv[1, 1] * y + H_inv[1, 2]
            sw = H_inv[2, 0] * x + H_inv[2, 1] * y + H_inv[2, 2]
            if abs(sw) < 1e-9:
                continue
            ix = sx / sw
            iy = sy / sw

            if ix < 0 or iy < 0 or ix > image.shape[1] - 1 or iy > image.shape[0] - 1:
                continue

            x0, y0 = int(ix), int(iy)
            x1, y1 = min(x0 + 1, image.shape[1] - 1), min(y0 + 1, image.shape[0] - 1)
            fx, fy = ix - x0, iy - y0

            for ch in range(image.shape[2]):
                top = image[y0, x0, ch] * (1 - fx) + image[y0, x1, ch] * fx
                bottom = image[y1, x0, ch] * (1 - fx) + image[y1, x1, ch] * fx
                out[y, x, ch] = int(round(top * (1 - fy) + bottom * fy))
    return out


def warp_affine_manual(image, M, out_w, out_h, border_value=255):
    """手写仿射 warp：把 2x3 矩阵扩成 3x3 后调用单应 warp。"""
    H = np.vstack([M, [0, 0, 1]])
    return warp_homography_manual(image, H, out_w, out_h, border_value)


In [ ]:
# 读取图像并手写三种几何变换
img = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
assert img is not None, "读取 lena.jpeg 失败"
h, w = img.shape[:2]

# 1) 相似变换：缩放 0.8、旋转 15 度、平移
M_sim = make_similarity_matrix(0.8, 15, 60, 40)
sim = warp_affine_manual(img, M_sim, w, h)

# 2) 仿射变换：非等比例缩放 + 剪切
M_aff = make_affine_matrix(1.2, 0.3, 40, -0.1, 1.0, 20)
aff = warp_affine_manual(img, M_aff, w, h)

# 3) 单应变换：透视效果
H_homo = make_homography_matrix(1.0, 0.0, 40, 0.0, 1.0, 40, -0.0008, 0.0003, 1.0)
homo = warp_homography_manual(img, H_homo, w, h)

# ---------- 与 OpenCV 对比验证 ----------
sim_cv = cv2.warpAffine(img, M_sim, (w, h), borderValue=(255, 255, 255))
aff_cv = cv2.warpAffine(img, M_aff, (w, h), borderValue=(255, 255, 255))
homo_cv = cv2.warpPerspective(img, H_homo, (w, h), borderValue=(255, 255, 255))
compare_results(sim, sim_cv, "相似变换")
compare_results(aff, aff_cv, "仿射变换")
compare_results(homo, homo_cv, "单应变换")

cv_imwrite("similarity_result.jpg", sim)
cv_imwrite("affine_result.jpg", aff)
cv_imwrite("homography_result.jpg", homo)

show_images([img, sim, aff, homo], ["原图", "相似变换", "仿射变换", "单应变换"], figsize=(14, 4))


In [ ]:
# 按老师要求：新建白色底图，把原图和变换结果放在同一张底图上，便于观察位置关系
canvas = np.full((h, w * 2 + 80, 3), 255, dtype=np.uint8)
canvas[:h, :w] = img
canvas[:sim.shape[0], w + 20:w + 20 + sim.shape[1]] = sim
cv_imwrite("similarity_comparison.jpg", canvas)
show_images([canvas], ["原图(左) + 相似变换(右) 白底对比"], figsize=(12, 4))


## 三、结果与参数分析

- 手写 warp 与 OpenCV 的 MAE 应接近 0（差异仅来自插值取整），这验证了反向映射 + 双线性插值的正确性。
- 相似变换保持物体形状，仅改变大小/朝向/位置；仿射变换会拉伸/剪切；单应变换会产生透视变形。
- 白底拼接用于直观比较变换前后两张图之间的**位置关系**。

**易错点**
1. 反向映射必须用**逆矩阵**，把输出坐标映射回输入坐标。
2. 插值前要检查浮点坐标是否落在输入图范围内，越界像素填白。
3. 双线性插值需处理边界（`x+1`、`y+1` 不能越界）。
4. 单应齐次坐标要除以 `w` 分量。


## 四、科研规范小结

1. **矩阵构造与 warp 分离**：变换矩阵显式构造，便于单测与复用。
2. **手写矩阵求逆**：体现对线性代数的掌握，也满足"不能调用函数库"的要求。
3. **数值对比**：用 OpenCV 的 warp 作为 ground truth 验证手写实现。


## 五、练习：绕图像中心旋转 45 度

**要求**：手写构造"先平移到中心、旋转、再平移回来"的相似变换，实现绕中心旋转 45°，并与 OpenCV 结果对比。


In [ ]:
# ==================== 练习解决方案 ====================
def rotate_around_center_manual(image, angle_deg):
    """手写绕图像中心旋转 angle_deg 度。"""
    h, w = image.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    a = math.radians(angle_deg)
    # 先平移到原点、旋转、再平移回中心（3x3 连乘后取前两行）
    M = np.array([
        [math.cos(a), -math.sin(a), cx - cx * math.cos(a) + cy * math.sin(a)],
        [math.sin(a),  math.cos(a), cy - cx * math.sin(a) - cy * math.cos(a)],
    ])
    return warp_affine_manual(image, M, w, h)

rot = rotate_around_center_manual(img, 45)
rot_cv = cv2.warpAffine(img, cv2.getRotationMatrix2D((w/2, h/2), 45, 1.0), (w, h), borderValue=(255,255,255))
compare_results(rot, rot_cv, "绕中心旋转45度")
show_images([img, rot], ["原图", "绕中心旋转45度"], figsize=(9, 4))
